In [ ]:
# pylint: disable=import-error, use-dict-literal, unused-import, missing-module-docstring

'\nCancer gene enrichment analysis using genomic interval intersections.\n\nThis notebook:\n1. Generates random control BED files (100 kb resolution) for null enrichment estimation.\n2. Formats COSMIC Cancer Gene Census (v103, GRCh38) regions into BED format,\n   with an optional subset restricted to amplification/deletion genes.\n3. Intersects:\n     - Important cancer-related SHAP feature regions\n     - Multiple random control region sets\n   against Cancer Gene Census regions using bedtools (≥2.31.0).\n4. Aggregates overlap counts and compares observed cancer-feature overlaps\n   to the empirical distribution from random regions.\n5. Computes summary statistics (mean, std, median, IQR, max) and a z-score\n   to quantify enrichment relative to random expectation.\n\nPurpose:\nAssess whether important epigenomic features are significantly enriched\nfor known cancer genes compared to matched random genomic regions.\n'

# Cancer gene enrichment analysis using genomic interval intersections.

**Goal**:
Assess whether important epigenomic features are significantly enriched for known cancer genes compared to matched random genomic regions.

This notebook:
1. Generates random control BED files (100 kb resolution) for null enrichment estimation.
2. Formats COSMIC Cancer Gene Census (v103, GRCh38) regions into BED format,
   with an optional subset restricted to amplification/deletion genes.
3. Intersects against Cancer Gene Census regions using bedtools:
     - Important cancer-related SHAP feature regions
     - Multiple random control region sets
4. Aggregates overlap counts and compares observed cancer-feature overlaps
   to the empirical distribution from random regions.
5. Computes summary statistics (mean, std, median, IQR, max) and a z-score
   to quantify enrichment relative to random expectation.



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm import tqdm

from epiclass.utils.bed_utils import create_new_random_bed

### Control beds for intersection results

In [ ]:
HDF5_SIZE_100KB = 30321
nb_random_regions = 336
resolution = 100 * 1000

n_beds = 200
output_dir = Path.home() / "Projects/epiclass/input/filter" / "random_n336"

# create_new_random_bed(
#     HDF5_SIZE_100KB,
#     nb_random_regions,
#     resolution,
#     output_dir=output_dir,
#     n_bed=n_beds,
#     sort=True,
# )

### Format cancer genes as bed

In [ ]:
paper_dir = Path.home() / "Projects/epiclass/output/paper"
regions_dir = paper_dir / "data" / "regions" / "cancer"

In [ ]:
region_name = "CancerGeneCensus"
filename = f"Cosmic_{region_name}_v103_GRCh38.tsv"

regions_path = regions_dir / filename
df = pd.read_csv(regions_path, sep="\t", header=0, low_memory=False)
print(df.shape)
display(df.head())

In [ ]:
# Can't compute enrichments for regions without genome coordinates
N_before = df.shape[0]
df = df[~df["GENOME_START"].isna()]
print(f"Excluding regions without genome coordinates: {N_before - df.shape[0]}")

In [ ]:
# Use conventional bed region naming
df["chr"] = "chr" + df["CHROMOSOME"].astype(str)

# alias
df["startpos"] = df["GENOME_START"].astype(int)
df["endpos"] = df["GENOME_STOP"].astype(int)

### Alt version

In [ ]:
display(df["MUTATION_TYPES"].value_counts())

In [ ]:
cancer_regions_ampdel = df.copy()
condition = cancer_regions_ampdel["MUTATION_TYPES"].str.contains("A|D", na=False)
print(f"Number of amp/del regions: {condition.sum()}")
cancer_regions_ampdel = cancer_regions_ampdel[condition]

In [ ]:
for df, region_name in zip([cancer_regions_ampdel, df], ["-AmpDel", ""]):
    df = df[["chr", "startpos", "endpos"]].copy()
    df.sort_values(["chr", "startpos", "endpos"], inplace=True)
    bed_path = regions_dir / Path(filename).with_name(
        f"Cosmic_CancerGeneCensus_v103_GRCh38{region_name}.bed"
    )
    df.to_csv(
        bed_path,
        sep="\t",
        index=False,
        header=False,
    )
    print(f"Wrote {df.shape[0]} regions to {bed_path}")

In [ ]:
selected_bed = regions_dir / "Cosmic_CancerGeneCensus_v103_GRCh38-AmpDel.bed"
region_name = "CancerGeneCensus-AmpDel"

#### Intersect with random

In [ ]:
output_dir = regions_dir / "overlap_SHAP"
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

Random beds intersect (using bedtools v2.31.0)

In [ ]:
random_output_dir = regions_dir / f"overlap_random_n{nb_random_regions}"
if not random_output_dir.exists():
    random_output_dir.mkdir(exist_ok=True)

In [ ]:
input_dir = Path.home() / "Projects/epiclass/input/filter" / "random_n336"

input_files = list(input_dir.glob("*random_n336*.bed"))
for input_bed_path in tqdm(input_files, desc="Processing BED files"):
    input_bed_name = input_bed_path.name.split(".")[1]

    output_path = random_output_dir / f"{input_bed_name}_intersect_{region_name}.tsv"

    subprocess.check_call(
        f"bedtools intersect -C -f 0.5 -a {input_bed_path} -b {selected_bed} > {output_path}",
        shell=True,
    )

Important cancer features bed intersect

In [ ]:
input_dir = (
    Path.home()
    / "Projects/epiclass/results/shap_analysis/join_important_features/hg38_100kb_all_none/global_info/cancer/"
)
input_bed_path = input_dir / "cancer_intersection_merge_samplings.bed"
output_path = output_dir / f"{input_bed_path.stem}_intersect_{region_name}.tsv"

subprocess.check_call(
    f"bedtools intersect -C -f 0.5 -a {input_bed_path} -b {selected_bed} > {output_path}",
    shell=True,
)

#### Compute statistics for specified features VS random features

In [ ]:
results_dict = {}

intersect_files = list(output_dir.glob(f"*intersect_{region_name}.tsv"))
if not intersect_files:
    raise FileNotFoundError("No intersect files found")
intersect_files.extend(list(random_output_dir.glob(f"*intersect_{region_name}.tsv")))

assert len(intersect_files) == n_beds + 1

In [ ]:
for intersect_file in intersect_files:
    # print(intersect_file)
    df = pd.read_csv(intersect_file, sep="\t", header=None)
    df.columns = ["chr", "startpos", "endpos", "nb_hits"]
    results_dict[str(intersect_file.stem)] = df["nb_hits"].sum()

In [ ]:
random_names = [set_name for set_name in results_dict if "random" in set_name]
n_beds = len(random_names)

# Compute the average of hits and stdev for random beds
all_random_hits = np.array([results_dict[name] for name in random_names])
random_mean_hits = np.mean(all_random_hits)
random_std_hits = np.std(all_random_hits)
random_max = np.max(all_random_hits)
random_iqr = np.percentile(all_random_hits, 75) - np.percentile(all_random_hits, 25)
random_median = np.median(all_random_hits)

print(
    f"Random beds (n={n_beds}): mean={random_mean_hits:.2f}, std={random_std_hits:.2f}, "
    f"median={random_median}, IQR={random_iqr}, max={random_max}"
)

In [ ]:
# Compare values of important cancer features bed with random beds
selected_name = [set_name for set_name in results_dict if "random" not in set_name][0]
cancer_hits = results_dict[selected_name]
z_score = (cancer_hits - random_mean_hits) / random_std_hits

print(
    f"{selected_name}: {cancer_hits} hits, rnd_mean: {random_mean_hits:.0f}, rnd_std: {random_std_hits:.1f}, z_score: {z_score:.2f}"
)